In [ ]:
import time
import xarray as xr
from dask_gateway import Gateway

import config
from preprocess import slice_domain, extract_surface_variable, rel_humidity_calc, extract_land_mask

# Gateway Startup

In [ ]:
gateway = Gateway()

In [ ]:
options = gateway.cluster_options()
options

In [ ]:
cluster = gateway.new_cluster(options)
cluster.scale(10)
client = cluster.get_client()
client

# ARCO Surface Variable Pipeline

In [ ]:
# FULL SEVEN SURFACE VARIABLE LOOP

ds = xr.open_zarr(config.ARCO_URL, chunks={"time": 24}, storage_options=dict(token="anon"))

for out_name, (arco_name, method) in config.ARCO_SURFACE.items():
    t0 = time.time()
    extract_surface_variable(ds, arco_name, out_name, method, config.SCRATCH)
    print(f"{out_name} done in {time.time()-t0:.1f}s")

In [ ]:
# VARIABLE VERIFICATION

for name in config.ARCO_SURFACE.keys():
    da = xr.open_zarr(f"{config.SCRATCH}/{name}.zarr")[name]
    n_nan = int(da.isnull().sum().compute())
    print(f"{name}: shape={da.shape}, "
          f"{str(da.time.values[0])[:10]}→{str(da.time.values[-1])[:10]}, "
          f"NaNs={n_nan}, range=[{float(da.min().compute()):.2f}, {float(da.max().compute()):.2f}]")

In [ ]:
# COMPUTING RELATIVE HUMIDITY

t2m = xr.open_zarr(f"{config.SCRATCH}/t2m.zarr")["t2m"]
d2m = xr.open_zarr(f"{config.SCRATCH}/d2m.zarr")["d2m"]

r = rel_humidity_calc(t2m, d2m).rename("r")
r = r.chunk({"time": 365, "latitude": -1, "longitude": -1})

out = r.to_dataset()
for var in out.variables:
    out[var].encoding.clear()

out.to_zarr(f"{config.SCRATCH}/r.zarr", mode="w")
print("r written.")

check = xr.open_zarr(f"{config.SCRATCH}/r.zarr")["r"]
print(f"r: shape={check.shape}, range=[{float(check.min().compute()):.2f}, {float(check.max().compute()):.2f}]")

In [ ]:
# CHECKING LOCAL SAVED FILES

expected_names = ["msl", "t2m", "u10", "v10", "tcc", "ssr", "r"]

for name in expected_names:
    path = f"{config.SCRATCH}/{name}.zarr"
    try:
        da = xr.open_zarr(path)[name]
        
        vmin = float(da.min().compute())
        vmax = float(da.max().compute())
        
        _ = float(da.isel(time=0).mean().compute())
        _ = float(da.isel(time=-1).mean().compute())
        print(f"{name}: Shape={da.shape}, Range=[{vmin:.2f}, {vmax:.2f}]")
    except Exception as e:
        print(f"{name}: Failed {type(e).__name__}: {str(e)[:80]}")

# Land-Sea Mask

In [ ]:
ds = xr.open_zarr(config.ARCO_URL, chunks=None, storage_options=dict(token="anon"))
extract_land_mask(ds, config.SCRATCH, threshold=0.5)

In [ ]:
land = xr.open_zarr(f"{config.SCRATCH}/land.zarr")["land"]
print(land.dims, land.shape, f"— land cells: {int(land.sum())}/{land.size}")

# ARCO Pressure Variable Pipeline

In [ ]:
ds = xr.open_zarr(config.ARCO_URL, chunks={"time": 24}, storage_options=dict(token="anon"))

arco_name, level = config.ARCO_PLEVEL["u"]

t0 = time.time()
da = ds[[arco_name]].sel(time=slice("1940-01-01", "1940-12-31"))
da = slice_domain(da)[arco_name]
da = da.sel(level=level)
daily = da.resample(time="1D").mean()
daily = daily.chunk({"time": 365, "latitude": -1, "longitude": -1})
out = daily.to_dataset().rename({arco_name: "u_probe"})
for var in out.variables:
    out[var].encoding.clear()
out.to_zarr(f"{config.SCRATCH}/_probe/u_1940.zarr", mode="w")
print(f"u 1940 done in {time.time()-t0:.1f}s")

# Gateway Shutdown

In [ ]:
print(gateway.list_clusters())

In [ ]:
for c in gateway.list_clusters():
    gateway.connect(c.name).shutdown()

In [ ]:
print(gateway.list_clusters())